# J022511-045433 · Region detection

Run the cells from top to bottom to execute this stage and display its scientific outputs.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config/systems.json").is_file())
sys.path.insert(0, str(ROOT))
from photo_lens import available_systems, run_stage, stage_artifacts
from IPython.display import display, Image, Markdown
SYSTEM = "J022511-045433"
record = next(item for item in available_systems() if item["id"] == SYSTEM)
record


## 1. Inspect the detection image
Source detection uses the exact AutoLens lens-light-subtracted image. Lens, source and background regions are stored in celestial coordinates.

In [ ]:
from astropy.io import fits
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
path = ROOT / "1.lens-modeling" / SYSTEM / "output/wcs_products_stage4" / f"{SYSTEM}_autolens_lens_light_subtracted_wcs.fits"
data = fits.getdata(path)
plt.figure(figsize=(7, 6))
plt.imshow(data, origin="lower", cmap="gray", norm=simple_norm(data, "asinh", percent=99))
plt.title(SYSTEM + " · lens-light-subtracted image")
plt.show()

## 2. Detect and construct apertures
SExtractor identifies components; point-reflection source fractions separate overlapping lens and source light. The background aperture is the point reflection of the source. SExtractor must be on PATH, or set `PHOTO_LENS_SEXTRACTOR` to its executable.

In [ ]:
artifacts = run_stage(SYSTEM, "regions")

## 3. Inspect the lens, source and symmetric background
Magenta: lens; cyan: source; amber: symmetric background.

In [ ]:
display(Image(filename=str(artifacts[0])))
from regions import Regions
folder = ROOT / "2.region-detection/regions" / SYSTEM
for kind in ["lens", "source", "background"]:
    regions = Regions.read(folder / f"{SYSTEM}_{kind}_point_fraction040.reg", format="ds9")
    print(kind, len(regions), "sky region(s)")